# A08 - Transfer Learning

Este notebook prepara e treina um pipeline de **transfer learning** sobre o
mesmo dataset usado nos artefatos anteriores:

- `artefatos/a02_baseline_classico/a02_baseline_classico.ipynb`
- `artefatos/a05_cnn_simples/a05_cnn_simples.ipynb`

O foco aqui é consolidar a etapa de dados no formato certo para backbones
pré-treinados:

- split `treino/validação/teste` com controle por `image_id`;
- conversão dos `pixel_*` para tensor 4D;
- `resize` espacial para o tamanho do backbone;
- normalização ajustada **apenas** no treino;
- `data augmentation` apenas no treino.

## Protocolo de dados

A lógica foi movida para o código reutilizável do projeto:

- `src/models/cnn_data_prep.py`
- `src/models/cnn_tf_data_pipeline.py`

Isso evita reimplementar manualmente no notebook a mesma etapa em cada artefato.

In [1]:
import importlib.util
import subprocess
import sys

PYTHON_VERSION = sys.version_info[:3]
REQUIRED_PACKAGES = {
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}

if PYTHON_VERSION < (3, 10) or PYTHON_VERSION >= (3, 13):
    raise RuntimeError(
        "Este notebook de A08 foi preparado para rodar com TensorFlow em Python 3.10, 3.11 ou 3.12. "
        f"Ambiente atual: {sys.version.split()[0]}. "
        "Use Google Colab ou um ambiente local com Python 3.11/3.12."
    )

REQUIRED_PACKAGES["tensorflow"] = "tensorflow"

missing = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Dependências OK.")

Dependências OK.


In [2]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tensorflow import keras
from tensorflow.keras import layers


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "src").exists():
        return cwd
    for parent in [cwd, *cwd.parents]:
        if (parent / "src").exists():
            return parent
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")


PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from models.cnn_builder import build_transfer_model
from models.cnn_data_prep import prepare_grouped_cnn_splits
from models.cnn_tf_data_pipeline import build_train_val_test_tf_data

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_CSV = PROJECT_ROOT / "data" / "pixels_dataset.csv"
EXTRACTED_CODES_JSON = PROJECT_ROOT / "data" / "extracted_codes.json"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "a08_transfer_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_SIZE = 0.20
VAL_SIZE = 0.20
BATCH_SIZE = 8
TARGET_SIZE = (160, 160)
NORMALIZATION = "zscore"
EPOCHS = 12
LEARNING_RATE = 1e-4
FINE_TUNE_LAST_LAYERS = 20

print(f"Projeto: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_CSV}")
print(f"Códigos: {EXTRACTED_CODES_JSON}")
print(f"Saída: {OUTPUT_DIR}")

Projeto: /Users/mateus/Projetos/Academico/Inteli/g01
Dataset: /Users/mateus/Projetos/Academico/Inteli/g01/data/pixels_dataset.csv
Códigos: /Users/mateus/Projetos/Academico/Inteli/g01/data/extracted_codes.json
Saída: /Users/mateus/Projetos/Academico/Inteli/g01/outputs/a08_transfer_learning


In [3]:
assert DATASET_CSV.exists(), f"Arquivo não encontrado: {DATASET_CSV}"
assert EXTRACTED_CODES_JSON.exists(), f"Arquivo não encontrado: {EXTRACTED_CODES_JSON}"

df = pd.read_csv(DATASET_CSV)

split_data = prepare_grouped_cnn_splits(
    df,
    extracted_codes_path=EXTRACTED_CODES_JSON,
    data_format="channels_last",
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    seed=SEED,
)

prep_summary = {
    "dataset_rows": int(len(df)),
    "valid_rows": int(split_data["split_meta"]["n_valid"]),
    "input_shape_raw": (
        int(split_data["shape_info"]["height"]),
        int(split_data["shape_info"]["width"]),
        int(split_data["shape_info"]["n_channels"]),
    ),
    "train_samples": int(split_data["split_meta"]["n_train"]),
    "val_samples": int(split_data["split_meta"]["n_val"]),
    "test_samples": int(split_data["split_meta"]["n_test"]),
    "train_images": int(len(split_data["split_meta"]["train_ids"])),
    "val_images": int(len(split_data["split_meta"]["val_ids"])),
    "test_images": int(len(split_data["split_meta"]["test_ids"])),
}

display(pd.DataFrame([prep_summary]))

,dataset_rows,valid_rows,input_shape_raw,train_samples,val_samples,test_samples,train_images,val_images,test_images
0,295,295,"(128, 128, 9)",177,59,59,177,59,59


In [4]:
tf_data = build_train_val_test_tf_data(
    split_data["X_train"],
    split_data["y_train"],
    split_data["X_val"],
    split_data["y_val"],
    split_data["X_test"],
    split_data["y_test"],
    batch_size=BATCH_SIZE,
    normalization=NORMALIZATION,
    resize_to=TARGET_SIZE,
    data_format="channels_last",
    target_channels=split_data["shape_info"]["n_channels"],
    augment_train=True,
    seed=SEED,
)

data_overview = pd.DataFrame(
    [
        {
            "split": "train",
            "samples": tf_data["train_meta"]["n_samples"],
            "input_shape": tf_data["train_meta"]["input_shape"],
            "augment": tf_data["train_meta"]["augment"],
        },
        {
            "split": "val",
            "samples": tf_data["val_meta"]["n_samples"],
            "input_shape": tf_data["val_meta"]["input_shape"],
            "augment": tf_data["val_meta"]["augment"],
        },
        {
            "split": "test",
            "samples": tf_data["test_meta"]["n_samples"],
            "input_shape": tf_data["test_meta"]["input_shape"],
            "augment": tf_data["test_meta"]["augment"],
        },
    ]
)
display(data_overview)

,split,samples,input_shape,augment
0,train,177,"(160, 160, 9)",True
1,val,59,"(160, 160, 9)",False
2,test,59,"(160, 160, 9)",False


## Decisão de arquitetura

O dataset ASTER possui `9` bandas, enquanto o `MobileNetV2` pré-treinado em
ImageNet espera `3` canais. Em vez de descartar bandas, o notebook usa uma
**camada adaptadora `1x1`** para projetar a entrada `9 → 3` antes do backbone.

In [5]:
model, model_info = build_transfer_model(
    input_shape=tf_data["train_meta"]["input_shape"],
    learning_rate=LEARNING_RATE,
    fine_tune_last_layers=FINE_TUNE_LAST_LAYERS,
)
model.summary()
model_info

Model: "a08_transfer_learning_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ aster_9ch_input (InputLayer)    │ (None, 160, 160, 9)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ channel_adapter (Conv2D)        │ (None, 160, 160, 3)    │            27 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ channel_adapter_bn              │ (None, 160, 160, 3)    │            12 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ channel_adapter_relu            │ (None, 160, 160, 3)    │             0 │
│ (Activation)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prediction (Dense)              │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,304 (8.62 MB)

 Trainable params: 1,196,194 (4.56 MB)

 Non-trainable params: 1,063,110 (4.06 MB)

{'backbone': 'MobileNetV2',
 'pretrained_loaded': True,
 'fine_tune_last_layers': 20,
 'input_shape': (160, 160, 9),
 'learning_rate': 0.0001,
 'dropout_rate': 0.25}